### This notebook analyzes FED3 Bandit data
<br>
<img src="https://fed3bandit.readthedocs.io/en/latest/_static/fed3bandit_logo1.svg" width="200" />


Authors: Chantelle Murrell and Sebastian Alves<br>
Updated: 12-16-25  
Version 1.1.0

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#!pip install pingouin
import sys
sys.path.insert(0, "src")           # relative to the project root where the notebook runs
from fedlib.extracted import fed3bandit_extracted, fed3_loading, fed3_fedframe
from fedlib.fedcore import core
from fedlib.fedassays import fedbandit

# Import other libraries (will have to kick out to top level)
import pandas as pd
import os
import zipfile
from pathlib import Path
from contextlib import ExitStack
import tempfile
import warnings

# for reloading packages
import importlib

In [3]:
# FMR1
ROOT_PATH = Path(r"C:\Users\william.b\Box\Kravitz Lab Box Drive\William\5_fed3_pipeline\TESTDIRDATA\FMR1\bandit100")

# define the directory for the L1 zipped data
L1_PATH = Path(ROOT_PATH, "FMR1_007_Bandit100_L1.zip")

# define the path for the key file
# the key file should always be a xlsx
KEY_PATH = Path(ROOT_PATH, "FMR1_007_Key.xlsx")

In [4]:
#@title Upload files
# reload packages
importlib.reload(core)

# fed3 warns on duplicate timestamps, but for FED data this is expected
# and not meaningful — suppress it globally rather than logging per-file
warnings.filterwarnings(
    "ignore",
    message="Index has duplicate values.*",
    category=RuntimeWarning,
)

# ========================== Call functions ============================ #
# ------ Ingest data ------ #
# call to lkoad lists and ingest the data from the l1 folder
fed_list, loaded_files, session_types = core.ingest_l1(L1_PATH)

# ------ Create Meta Data Key ------ #
# upload key file
key_df, msg = core._read_key_from_upload(KEY_PATH)

# build key dataframe
key_df2 = core.build_or_rematch_key_df(loaded_files, session_types, key_df, msg_hint = f"Key status: {msg}")


==> Beginning Ingest
  [ok] Ingest Complete - Loaded 46 files. Session types captured for all.
==> Reading in key file
  [ok] Done - Key loaded
==> Building key dataframe
  [ok] Key status: Key loaded from upload (FMR1_007_Key.xlsx). 'Mouse_ID' found.
  [ok] Merged key columns into Key_Df (46 rows, 27 cols).


In [5]:
# Begin Bandit specific pipeline
importlib.reload(fedbandit)

# ------ Plot individual Bandit Data ------ #

# Clean the filename column
metadata_df = fedbandit.clean_key_filename_column(key_df2)

# Create the individual Bandit Plot
plt_list = fedbandit._plot_file_core(fed_list, metadata_df, ROOT_PATH)

print(plt_list)

==> Beginning to plot bandit assays
  [ok] saved FMR1_007_220_7_pLeft
  [ok] saved FMR1_007_220_9_pLeft
  [ok] saved FMR1_007_211_86_pLeft
  [ok] saved FMR1_007_212_88_pLeft
  [ok] saved FMR1_007_212_89_pLeft
  [ok] saved FMR1_007_211_82_pLeft
  [ok] saved FMR1_007_211_84_pLeft
  [ok] saved FMR1_007_217_58_pLeft
  [ok] saved FMR1_007_216_73_pLeft
  [ok] saved FMR1_007_216_72_pLeft
  [ok] saved FMR1_007_210_79_pLeft
  [ok] saved FMR1_007_221_47_pLeft
  [ok] saved FMR1_007_221_51_pLeft
  [ok] saved FMR1_007_221_50_pLeft
  [ok] saved FMR1_007_213_39_pLeft
  [ok] saved FMR1_007_222_19_pLeft
  [ok] saved FMR1_007_214_98_pLeft
  [ok] saved FMR1_007_218_77_pLeft
  [ok] saved FMR1_007_214_94_pLeft
  [ok] saved FMR1_007_219_23_pLeft
  [ok] saved FMR1_007_215_5_pLeft
  [ok] saved FMR1_007_222_18_pLeft
  [ok] saved FMR1_007_219_29_pLeft
  [ok] saved FMR1_007_217_55_pLeft
  [ok] saved FMR1_007_219_27_pLeft
  [ok] saved FMR1_007_217_52_pLeft
  [ok] saved FMR1_007_221_48_pLeft
  [ok] saved FMR1_007_

[WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_220_7_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_220_9_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_211_86_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_212_88_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_212_89_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/indv_bandit_plots/FMR1_007_211_82_pLeft'), WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIR

In [6]:
#@title Analyse bandit metrics - create the L3
importlib.reload(fedbandit)
importlib.reload(core)



### ------ Begin Analysis to build L3 ------ ###
### Generate cropped key for bandit use
#md = fedbandit.build_bandit_metakey(metadata_df)
meta_cols, md = core.build_metakey(metadata_df, assay = "bandit")

print(meta_cols)


### Identify key columns
id_col, other_id = core.pick_match_method(md)

# *** Weight column in this is all NA ****
### Compute bandit metrics
bm = fedbandit.compute_bandit_metrics(fed_list, md, id_col)

### Attach metadata to metrics 
bm_md = core.attach_meta(bm, md, id_col)

### Prepare and output L3 ###
bm_l3 = core.output_l3(bm_md, id_col, other_id, meta_cols, ROOT_PATH, assay = "bandit")

==> Computing Bandit metrics


['Genotype', 'Gene', 'Strain', 'Sex', 'Diet', 'Treatment', 'Condition', 'Pellets', 'Litter', 'Sire_Geno', 'Dam_Geno']


100%|██████████| 46/46 [00:02<00:00, 17.93it/s]
  [ok] metrics computed:
  [preview] Bandit metrics preview: — 46 rows x 21 cols
                           count     mean     std      min      25%      50%      75%       max
    PeakAccuracy            46.0    0.755   0.064    0.584    0.715    0.746    0.805     0.888
    Total_pellets           46.0  505.130  54.868  395.000  462.750  513.500  543.250   615.000
    Total_pokes             46.0  857.217  90.078  697.000  786.000  866.500  927.250  1062.000
    PokesPerPellet          46.0    1.703   0.133    1.409    1.623    1.702    1.793     2.091
    RetrievalTime           46.0    1.112   0.347    0.490    0.857    1.115    1.325     1.960
    PokeTime                46.0    0.141   0.033    0.090    0.120    0.140    0.160     0.230
    Win-stay                46.0    0.751   0.081    0.530    0.697    0.764    0.814     0.921
    Lose-shift              46.0    0.469   0.076    0.293    0.415    0.474    0.514     0.609
    Dai

In [7]:
# plotting 


importlib.reload(fedbandit)


### ------ Prepare Metric Barplots ------ ###
# plottings used bandit merged dataframe (bm_md) NOT bm_l3

### Build dataframe linking filenames to groupings 
mapped_df = core.build_group_selections(md)

### attach the grouping dataframe to the metrics
bm_plot = core.merge_group_selections(bm_md, mapped_df)

### Melt the metric dataframe to long format
bm_long = core.melt_metric(bm_plot, assay="bandit")

### Create variables to control aesthetics
x_checks, x_colors, ordered_x = core.define_aesthetics(bm_long)


### Actually create the plots
barplot_paths = core._run_plots(bm_long, x_checks, x_colors, ordered_x, ROOT_PATH)




==> Building group selection dataframe
  [ok] X-axis grouping (hierarchy): ['Genotype']
  [ok] Total unique files (X map): 46
  [table] table — 4 rows x 1 cols
           UniqueFiles
    Group             
    WT              15
    HET             11
    HEMI            10
    HOM             10
  [ok] Hue grouping: ['Sex']
  [ok] Total unique files (Hue map): 46
  [table] table — 2 rows x 1 cols
           UniqueFiles
    Group             
    F               27
    M               19
  [ok] Grouping dataframe built
==> Merging group df and metric df
  [ok] Group and Metric dataframe merged
==> Melting L3 to long format
  [ok] Metric dataframe Melted
==> Defining Aesthetics for Plotting
  [ok] Aesthetics defined - x_checks: {'WT': Checkbox(value=True, description='WT', indent=False, layout=Layout(width='260px')), 'HEMI': Checkbox(value=True, description='HEMI', indent=False, layout=Layout(width='260px')), 'HET': Checkbox(value=True, description='HET', indent=False, layout=Layout(wid

In [8]:
importlib.reload(fedbandit)

### ------- Plot Peak Accuracy ------ ###

# build dictionary of filenames linked to groups
basenames, xgroups = fedbandit.prep_pa_groups(mapped_df)


# build reverse dataframe
rev_df = fedbandit.build_rev_df(fed_list, xgroups, basenames)
#print(rev_df)

rev_plot_path = fedbandit.plot_rev_learning(rev_df, x_colors, ordered_x, bm_md, ROOT_PATH)



==> creating reverse learning plot
  [ok] Reverse learning plot created


In [26]:
importlib.reload(fedbandit)
importlib.reload(core)

fedbandit.assemble_bandit_l4(bm_long, rev_df, x_colors, ordered_x, bm_md, ROOT_PATH, fed_list, metadata_df, bandittype = "100")

==> Assembling L4 composite figure
  [ok] L4 example pair -> WT: FMR1_007_221_51 | FMR1: FMR1_007_307_43 (acc gap 0.060, pellet gap 13.0)
c:\Users\william.b\AppData\Local\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '
c:\Users\william.b\AppData\Local\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 1, but rank is 0
  warnings.warn('covariance of constraints does not have full '
c:\Users\william.b\AppData\Local\anaconda3\Lib\site-packages\statsmodels\base\model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J
c:\Users\william.b\AppData\Local\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

WindowsPath('C:/Users/william.b/Box/Kravitz Lab Box Drive/William/5_fed3_pipeline/TESTDIRDATA/FMR1/bandit100/L4/FMR1_L4.svg')

In [16]:
from fedlib.fedassays import fedbandit
from pathlib import Path

importlib.reload(fedbandit)
importlib.reload(core)

root = Path(r"C:\Users\william.b\Box\Kravitz Lab Box Drive\William\5_fed3_pipeline\TESTDIRDATA\MYT1L\bandit100")
res = fedbandit.run_bandit_l1_l4(root / "MYT1L_001_Bandit100_L1.zip",
                                 root / "MYT1L_001_Key.xlsx", root, bandittype = "100")

root = Path(r"C:\Users\william.b\Box\Kravitz Lab Box Drive\William\5_fed3_pipeline\TESTDIRDATA\FMR1\bandit100")
res = fedbandit.run_bandit_l1_l4(root / "FMR1_007_Bandit100_L1.zip",
                                 root / "FMR1_007_Key.xlsx", root, bandittype = "100")

==> Beginning Ingest
  [ok] Ingest Complete - Loaded 39 files. Session types captured for all.
==> Reading in key file
  [ok] Done - Key loaded
==> Building key dataframe
  [ok] Key status: Key loaded from upload (MYT1L_001_Key.xlsx). 'Mouse_ID' found.
  [ok] Merged key columns into Key_Df (39 rows, 25 cols).
==> Beginning to plot bandit assays
  [ok] saved MYT1L_001_9_14_pLeft
  [ok] saved MYT1L_001_5_24_pLeft
  [ok] saved MYT1L_001_5_28_pLeft
  [ok] saved MYT1L_001_2_5_pLeft
  [ok] saved MYT1L_001_2_8_pLeft
  [ok] saved MYT1L_001_9_12_pLeft
  [ok] saved MYT1L_001_9_13_pLeft
  [ok] saved MYT1L_001_8_39_pLeft
  [ok] saved MYT1L_001_12_46_pLeft
  [ok] saved MYT1L_001_12_51_pLeft
  [ok] saved MYT1L_001_1_1_pLeft
  [ok] saved MYT1L_001_5_23_pLeft
  [ok] saved MYT1L_001_5_27_pLeft
  [ok] saved MYT1L_001_1_4_pLeft
  [ok] saved MYT1L_001_2_6_pLeft
  [ok] saved MYT1L_001_9_11_pLeft
  [ok] saved MYT1L_001_7_32_pLeft
  [ok] saved MYT1L_001_8_41_pLeft
  [ok] saved MYT1L_001_10_42_pLeft
  [ok] sa